# SMAX checkpoint evaluation

This notebook evaluates saved SMAX checkpoints with deterministic rollouts.

- Per-seed win rate is computed from multiple evaluation episodes.
- Across-seed mean and std are seed-level statistics, not episode-level statistics.
- Each map cell evaluates all constraints found for that map, and each constraint is compared across all supported models.
- You can switch between `best` and `final` checkpoints from the setup cell.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "main.py").is_file() and (candidate / "figures").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the pci_marl repo root from the current notebook working directory.")


ROOT = resolve_repo_root()
FIGURES_DIR = ROOT / "figures"
for path in (ROOT, FIGURES_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from smax_checkpoint_eval_utils import (
    PROJECT_FILTER,
    MAP_ORDER,
    bootstrap_project,
    discover_runs,
    evaluate_map,
    format_cross_map_latex,
    save_summary_tables,
)

DEVICE = "cpu"
CHECKPOINT_CHOICE = "best"  # switch to "final" when you want final checkpoints
EVAL_EPISODES = 64  # choose any value in the 32-100 range
TARGET_MAPS = MAP_ORDER.copy()
MAP_RESULTS = {}

bootstrap_project(DEVICE)
runs_df = discover_runs(ROOT / "checkpoints_for_eval", project_filter=PROJECT_FILTER)
print(f"Repo root: {ROOT}")
print(f"Loaded {len(runs_df)} evaluation runs from {PROJECT_FILTER}")
display(
    runs_df.groupby(["map_name", "constraint_label", "model_label"]).size().unstack(fill_value=0)
)


Repo root: /home/chamjin/research/pci_marl
Loaded 59 evaluation runs from pci-marl-smax-jax-v3


model_label                      ippo  mappo  pci-ippo
map_name       constraint_label                       
2s3z           tar_w=3              3      3         3
               tar_w=4              3      3         3
               vanilla              3      3         3
3m             tar_w=2              3      4         3
               vanilla              3      3         1
3s5z           tar_w=6              3      3         0
               vanilla              0      1         0
smacv2_5_units tar_w=3              3      0         0
               tar_w=4              3      0         0
               vanilla              2      0         0

## Per-map evaluation

Run the cells below one by one. Each cell evaluates every model available for a single map, but the evaluation is split by constraint so only runs trained under the same map/constraint pair are compared.

In [2]:
MAP_RESULTS = {}
for map_name in TARGET_MAPS:
    result = evaluate_map(
        runs_df,
        map_name,
        checkpoint_choice=CHECKPOINT_CHOICE,
        device=DEVICE,
        eval_episodes=EVAL_EPISODES,
    )
    if result["model_summary"].empty:
        print(f"Skipping {map_name}: no discovered runs")
        continue
    MAP_RESULTS[map_name] = result
    print(f"\n=== {map_name} ===")
    display(result["seed_summary"])
    display(result["model_summary"])
    print(result["latex"])


[SMAXWrapper] 시나리오 '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=1
[SMAXBatchWrapper] obs_dim=75, action_dim=8
[SMAXBatchWrapper] 3v3 × 1 환경 (vmap 병렬)
[SMAXBatchWrapper] 배치 JIT warm-up 시작...
[SMAXBatchWrapper]   에피소드 1/3 (0.0s)
[SMAXBatchWrapper]   에피소드 2/3 (5.5s)
[SMAXBatchWrapper]   에피소드 3/3 (5.5s)
[SMAXBatchWrapper] warm-up 완료! (총 5.5초)
[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78
Environment name: SMAXBatchWrapper
use_delta_critic: False
[SMAXWrapper] 시나리오 '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=1
[SMAXBatchWrapper] obs_dim=75, action_dim=8
[SMAXBatchWrapper] 3v3 × 1 환경 (vmap 병렬)
[SMAXBatchWrapper] 배치 JIT warm-up 시작...
[SMAXBatchWrapper]   에피소드 1/3 (0.0s)
[SMAXBatchWrapper]   에피소드 2/3 (6.1s)
[SMAXBatchWrapper]   에피소드 3/3 (6.1s)
[SMAXBatchWrapper] warm-up 완료! (총 6.2초)
[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78
Environment name: SMAXBatchWrapper
use_delta_critic: False
[SMAXWrapper] 시나리오 '3m': 3

,map_name,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs
0,3m,vanilla,pci-ippo,42,0.812500,1.771875,26.265625,1
1,3m,vanilla,mappo,1,0.000000,0.067708,20.343750,1
2,3m,vanilla,mappo,2,0.625000,1.570833,15.656250,1
3,3m,vanilla,mappo,42,0.765625,1.726042,17.515625,1
4,3m,vanilla,ippo,1,0.828125,1.806250,18.546875,1
5,3m,vanilla,ippo,2,0.000000,0.031250,18.328125,1
6,3m,vanilla,ippo,42,0.828125,1.806250,17.140625,1
7,3m,tar_w=2,pci-ippo,1,0.531250,1.417708,27.812500,1
8,3m,tar_w=2,pci-ippo,2,0.406250,0.993229,27.804688,2
9,3m,tar_w=2,mappo,1,0.281250,1.130208,15.140625,1


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,3m,vanilla,pci-ippo,0.812500,0.000000,1.771875,NaN,26.265625,NaN,1,1
1,3m,vanilla,mappo,0.463542,0.407550,1.121528,0.915928,17.838542,2.360375,3,3
2,3m,vanilla,ippo,0.552083,0.478118,1.214583,1.024797,18.005208,0.756698,3,3
3,3m,tar_w=2,pci-ippo,0.468750,0.088388,1.205469,0.300152,27.808594,0.005524,2,3
4,3m,tar_w=2,mappo,0.281250,0.281250,0.860764,0.758653,15.630208,0.517594,3,4
5,3m,tar_w=2,ippo,0.119792,0.207485,0.470139,0.658437,18.723958,3.292478,3,3


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 3m using best checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_3m_best}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
vanilla & -- & 81.2 $\pm$ 0.0 & 46.4 $\pm$ 40.8 & 55.2 $\pm$ 47.8 \\
tar_w=2 & -- & 46.9 $\pm$ 8.8 & 28.1 $\pm$ 28.1 & 12.0 $\pm$ 20.7 \\
\bottomrule
\end{tabular}
\end{table}
[SMAXWrapper] 시나리오 '2s3z': 5 allies vs 5 enemies
[SMAXBatchWrapper] '2s3z': 5v5, n_envs=1
[SMAXBatchWrapper] obs_dim=127, action_dim=10
[SMAXBatchWrapper] 5v5 × 1 환경 (vmap 병렬)
[SMAXBatchWrapper] 배치 JIT warm-up 시작...
[SMAXBatchWrapper]   에피소드 1/3 (0.0s)
[SMAXBatchWrapper]   에피소드 2/3 (7.8s)
[SMAXBatchWrapper]   에피소드 3/3 (7.9s)
[SMAXBatchWrapper] warm-up 완료! (총 7.9초)
[INFO] add_agent_id enabled: base_obs_dim=127, effective_obs_dim=132
Environment name: SMAXBatchWrapper
use_delta_critic: False
[SMAXWrapper] 시나리오 

,map_name,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs
0,2s3z,vanilla,pci-ippo,1,0.984375,1.982910,70.265625,1
1,2s3z,vanilla,pci-ippo,2,0.906250,1.897637,69.187500,1
2,2s3z,vanilla,pci-ippo,42,0.890625,1.883223,66.687500,1
3,2s3z,vanilla,mappo,1,0.875000,1.863887,70.609375,1
4,2s3z,vanilla,mappo,2,1.000000,2.000000,56.031250,1
5,2s3z,vanilla,mappo,42,0.843750,1.832500,72.312500,1
6,2s3z,vanilla,ippo,1,0.812500,1.782311,75.109375,1
7,2s3z,vanilla,ippo,2,0.468750,1.399581,73.921875,1
8,2s3z,vanilla,ippo,42,0.484375,1.400468,76.812500,1
9,2s3z,tar_w=3,pci-ippo,1,0.921875,1.918008,69.453125,1


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,2s3z,vanilla,pci-ippo,0.927083,0.050227,1.921256,0.053878,68.713542,1.835544,3,3
1,2s3z,vanilla,mappo,0.906250,0.082680,1.898795,0.089039,66.317708,8.948943,3,3
2,2s3z,vanilla,ippo,0.588542,0.194111,1.527453,0.220714,75.281250,1.452957,3,3
3,2s3z,tar_w=3,pci-ippo,0.338542,0.507352,1.167556,0.673193,66.572917,9.542697,3,3
4,2s3z,tar_w=3,mappo,0.427083,0.343158,1.316512,0.458210,70.281250,3.415233,3,3
5,2s3z,tar_w=3,ippo,0.270833,0.162630,1.167317,0.194410,78.703125,7.090652,3,3
6,2s3z,tar_w=4,pci-ippo,0.630208,0.183108,1.585255,0.214792,72.036458,3.485122,3,3
7,2s3z,tar_w=4,mappo,0.739583,0.231932,1.705060,0.273184,68.697917,3.321198,3,3
8,2s3z,tar_w=4,ippo,0.953125,0.031250,1.947970,0.036445,63.401042,1.349672,3,3


... (3 outputs left out)

\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 3s5z using best checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_3s5z_best}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
vanilla & -- & -- & 81.2 $\pm$ 0.0 & -- \\
tar_w=6 & -- & -- & 33.3 $\pm$ 20.6 & 0.0 $\pm$ 0.0 \\
\bottomrule
\end{tabular}
\end{table}
[SMAXWrapper] 시나리오 'smacv2_5_units': 5 allies vs 5 enemies
[SMAXBatchWrapper] 'smacv2_5_units': 5v5, n_envs=1
[SMAXBatchWrapper] obs_dim=127, action_dim=10
[SMAXBatchWrapper] 5v5 × 1 환경 (vmap 병렬)
[SMAXBatchWrapper] 배치 JIT warm-up 시작...
[SMAXBatchWrapper]   에피소드 1/3 (0.0s)
[SMAXBatchWrapper]   에피소드 2/3 (9.9s)
[SMAXBatchWrapper]   에피소드 3/3 (9.9s)
[SMAXBatchWrapper] warm-up 완료! (총 10.0초)
[INFO] add_agent_id enabled: base_obs_dim=127, effective_obs_dim=132
Environment name: SMAXBatchWrapper
use_delta_critic: False
[SMAXWrapper] 시나리오 'smacv2_5_units

,map_name,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs
0,smacv2_5_units,vanilla,ippo,1,0.515625,1.376327,44.468750,1
1,smacv2_5_units,vanilla,ippo,42,0.593750,1.483116,45.218750,1
2,smacv2_5_units,tar_w=3,ippo,1,0.078125,0.786328,46.328125,1
3,smacv2_5_units,tar_w=3,ippo,2,0.046875,0.662907,53.578125,1
4,smacv2_5_units,tar_w=3,ippo,42,0.000000,0.095745,24.734375,1
5,smacv2_5_units,tar_w=4,ippo,1,0.359375,1.158804,44.578125,1
6,smacv2_5_units,tar_w=4,ippo,2,0.312500,1.085805,40.671875,1
7,smacv2_5_units,tar_w=4,ippo,42,0.000000,0.104407,25.218750,1


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,smacv2_5_units,vanilla,ippo,0.554688,0.055243,1.429721,0.075511,44.843750,0.530330,2,2
1,smacv2_5_units,tar_w=3,ippo,0.041667,0.039322,0.514993,0.368286,41.546875,15.004524,3,3
2,smacv2_5_units,tar_w=4,ippo,0.223958,0.195365,0.783005,0.588816,36.822917,10.237539,3,3


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on SMACv2-5U using best checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_smacv2_5_units_best}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
vanilla & -- & -- & -- & 55.5 $\pm$ 5.5 \\
tar_w=3 & -- & -- & -- & 4.2 $\pm$ 3.9 \\
tar_w=4 & -- & -- & -- & 22.4 $\pm$ 19.5 \\
\bottomrule
\end{tabular}
\end{table}


In [3]:
if "2s3z" not in MAP_RESULTS:
    raise RuntimeError("Run the consolidated evaluation cell above first.")

display(MAP_RESULTS["2s3z"]["seed_summary"])
display(MAP_RESULTS["2s3z"]["model_summary"])
print(MAP_RESULTS["2s3z"]["latex"])


,map_name,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs
0,2s3z,vanilla,pci-ippo,1,0.984375,1.982910,70.265625,1
1,2s3z,vanilla,pci-ippo,2,0.906250,1.897637,69.187500,1
2,2s3z,vanilla,pci-ippo,42,0.890625,1.883223,66.687500,1
3,2s3z,vanilla,mappo,1,0.875000,1.863887,70.609375,1
4,2s3z,vanilla,mappo,2,1.000000,2.000000,56.031250,1
5,2s3z,vanilla,mappo,42,0.843750,1.832500,72.312500,1
6,2s3z,vanilla,ippo,1,0.812500,1.782311,75.109375,1
7,2s3z,vanilla,ippo,2,0.468750,1.399581,73.921875,1
8,2s3z,vanilla,ippo,42,0.484375,1.400468,76.812500,1
9,2s3z,tar_w=3,pci-ippo,1,0.921875,1.918008,69.453125,1


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,2s3z,vanilla,pci-ippo,0.927083,0.050227,1.921256,0.053878,68.713542,1.835544,3,3
1,2s3z,vanilla,mappo,0.906250,0.082680,1.898795,0.089039,66.317708,8.948943,3,3
2,2s3z,vanilla,ippo,0.588542,0.194111,1.527453,0.220714,75.281250,1.452957,3,3
3,2s3z,tar_w=3,pci-ippo,0.338542,0.507352,1.167556,0.673193,66.572917,9.542697,3,3
4,2s3z,tar_w=3,mappo,0.427083,0.343158,1.316512,0.458210,70.281250,3.415233,3,3
5,2s3z,tar_w=3,ippo,0.270833,0.162630,1.167317,0.194410,78.703125,7.090652,3,3
6,2s3z,tar_w=4,pci-ippo,0.630208,0.183108,1.585255,0.214792,72.036458,3.485122,3,3
7,2s3z,tar_w=4,mappo,0.739583,0.231932,1.705060,0.273184,68.697917,3.321198,3,3
8,2s3z,tar_w=4,ippo,0.953125,0.031250,1.947970,0.036445,63.401042,1.349672,3,3


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 2s3z using best checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_2s3z_best}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
vanilla & -- & 92.7 $\pm$ 5.0 & 90.6 $\pm$ 8.3 & 58.9 $\pm$ 19.4 \\
tar_w=3 & -- & 33.9 $\pm$ 50.7 & 42.7 $\pm$ 34.3 & 27.1 $\pm$ 16.3 \\
tar_w=4 & -- & 63.0 $\pm$ 18.3 & 74.0 $\pm$ 23.2 & 95.3 $\pm$ 3.1 \\
\bottomrule
\end{tabular}
\end{table}


In [4]:
if "3s5z" not in MAP_RESULTS:
    raise RuntimeError("Run the consolidated evaluation cell above first.")

display(MAP_RESULTS["3s5z"]["seed_summary"])
display(MAP_RESULTS["3s5z"]["model_summary"])
print(MAP_RESULTS["3s5z"]["latex"])


,map_name,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs
0,3s5z,vanilla,mappo,42,0.812500,1.795423,74.906250,1
1,3s5z,tar_w=6,mappo,1,0.109375,0.958060,77.765625,1
2,3s5z,tar_w=6,mappo,2,0.515625,1.455935,76.015625,1
3,3s5z,tar_w=6,mappo,42,0.375000,1.297366,81.421875,1
4,3s5z,tar_w=6,ippo,1,0.000000,0.674819,64.687500,1
5,3s5z,tar_w=6,ippo,2,0.000000,0.722817,70.062500,1
6,3s5z,tar_w=6,ippo,42,0.000000,0.099049,46.187500,1


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,3s5z,vanilla,mappo,0.812500,0.000000,1.795423,NaN,74.906250,NaN,1,1
1,3s5z,tar_w=6,mappo,0.333333,0.206305,1.237120,0.254346,78.401042,2.758569,3,3
2,3s5z,tar_w=6,ippo,0.000000,0.000000,0.498895,0.347107,60.312500,12.524351,3,3


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 3s5z using best checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_3s5z_best}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
vanilla & -- & -- & 81.2 $\pm$ 0.0 & -- \\
tar_w=6 & -- & -- & 33.3 $\pm$ 20.6 & 0.0 $\pm$ 0.0 \\
\bottomrule
\end{tabular}
\end{table}


In [5]:
if "smacv2_5_units" not in MAP_RESULTS:
    raise RuntimeError("Run the consolidated evaluation cell above first.")

display(MAP_RESULTS["smacv2_5_units"]["seed_summary"])
display(MAP_RESULTS["smacv2_5_units"]["model_summary"])
print(MAP_RESULTS["smacv2_5_units"]["latex"])


,map_name,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs
0,smacv2_5_units,vanilla,ippo,1,0.515625,1.376327,44.468750,1
1,smacv2_5_units,vanilla,ippo,42,0.593750,1.483116,45.218750,1
2,smacv2_5_units,tar_w=3,ippo,1,0.078125,0.786328,46.328125,1
3,smacv2_5_units,tar_w=3,ippo,2,0.046875,0.662907,53.578125,1
4,smacv2_5_units,tar_w=3,ippo,42,0.000000,0.095745,24.734375,1
5,smacv2_5_units,tar_w=4,ippo,1,0.359375,1.158804,44.578125,1
6,smacv2_5_units,tar_w=4,ippo,2,0.312500,1.085805,40.671875,1
7,smacv2_5_units,tar_w=4,ippo,42,0.000000,0.104407,25.218750,1


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,smacv2_5_units,vanilla,ippo,0.554688,0.055243,1.429721,0.075511,44.843750,0.530330,2,2
1,smacv2_5_units,tar_w=3,ippo,0.041667,0.039322,0.514993,0.368286,41.546875,15.004524,3,3
2,smacv2_5_units,tar_w=4,ippo,0.223958,0.195365,0.783005,0.588816,36.822917,10.237539,3,3


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on SMACv2-5U using best checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_smacv2_5_units_best}
\begin{tabular}{lcccc}
\toprule
Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
vanilla & -- & -- & -- & 55.5 $\pm$ 5.5 \\
tar_w=3 & -- & -- & -- & 4.2 $\pm$ 3.9 \\
tar_w=4 & -- & -- & -- & 22.4 $\pm$ 19.5 \\
\bottomrule
\end{tabular}
\end{table}


In [6]:
if not MAP_RESULTS:
    raise RuntimeError("Run the consolidated evaluation cell above first.")

combined_model_summary = pd.concat(
    [MAP_RESULTS[map_name]["model_summary"] for map_name in TARGET_MAPS if map_name in MAP_RESULTS],
    ignore_index=True,
)
display(combined_model_summary)

latex_code = format_cross_map_latex(
    combined_model_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
)
print(latex_code)

paths = save_summary_tables(
    combined_model_summary,
    FIGURES_DIR,
    checkpoint_choice=CHECKPOINT_CHOICE,
)
print(f"Saved CSV: {paths['csv']}")
print(f"Saved TeX: {paths['tex']}")


,map_name,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs
0,3m,vanilla,pci-ippo,0.812500,0.000000,1.771875,NaN,26.265625,NaN,1,1
1,3m,vanilla,mappo,0.463542,0.407550,1.121528,0.915928,17.838542,2.360375,3,3
2,3m,vanilla,ippo,0.552083,0.478118,1.214583,1.024797,18.005208,0.756698,3,3
3,3m,tar_w=2,pci-ippo,0.468750,0.088388,1.205469,0.300152,27.808594,0.005524,2,3
4,3m,tar_w=2,mappo,0.281250,0.281250,0.860764,0.758653,15.630208,0.517594,3,4
5,3m,tar_w=2,ippo,0.119792,0.207485,0.470139,0.658437,18.723958,3.292478,3,3
6,2s3z,vanilla,pci-ippo,0.927083,0.050227,1.921256,0.053878,68.713542,1.835544,3,3
7,2s3z,vanilla,mappo,0.906250,0.082680,1.898795,0.089039,66.317708,8.948943,3,3
8,2s3z,vanilla,ippo,0.588542,0.194111,1.527453,0.220714,75.281250,1.452957,3,3
9,2s3z,tar_w=3,pci-ippo,0.338542,0.507352,1.167556,0.673193,66.572917,9.542697,3,3


\begin{table*}[t]
\centering
\caption{Deterministic SMAX win rate using best checkpoints. Rows are grouped by map and training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_cross_map_best}
\resizebox{\linewidth}{!}{%
\begin{tabular}{llcccc}
\toprule
Map & Constraint & pci-mappo & pci-ippo & mappo & ippo \\
\midrule
3m & vanilla & -- & 81.2 $\pm$ 0.0 & 46.4 $\pm$ 40.8 & 55.2 $\pm$ 47.8 \\
3m & tar_w=2 & -- & 46.9 $\pm$ 8.8 & 28.1 $\pm$ 28.1 & 12.0 $\pm$ 20.7 \\
2s3z & vanilla & -- & 92.7 $\pm$ 5.0 & 90.6 $\pm$ 8.3 & 58.9 $\pm$ 19.4 \\
2s3z & tar_w=3 & -- & 33.9 $\pm$ 50.7 & 42.7 $\pm$ 34.3 & 27.1 $\pm$ 16.3 \\
2s3z & tar_w=4 & -- & 63.0 $\pm$ 18.3 & 74.0 $\pm$ 23.2 & 95.3 $\pm$ 3.1 \\
3s5z & vanilla & -- & -- & 81.2 $\pm$ 0.0 & -- \\
3s5z & tar_w=6 & -- & -- & 33.3 $\pm$ 20.6 & 0.0 $\pm$ 0.0 \\
SMACv2-5U & vanilla & -- & -- & -- & 55.5 $\pm$ 5.5 \\
SMACv2-5U & tar_w=3 & -- & -- & -- & 4.2 $\pm$ 3.9 \\
SMACv2-5U & tar_w=4 & -- & -- & -- & 22.4 $\pm$ 